### Introduction

An organization wants to predict who possible defaulters are for the consumer loans product. They have data about historic customer behavior based on what they have observed. Hence when they acquire new customers they want to predict who is riskier and who is not.

#### Task Details
An organization wants to predict who possible defaulters are for the consumer loans product. They have data about historic customer behavior based on what they have observed. Hence when they acquire new customers they want to predict who is riskier and who is not.

#### What do you have to do?
You are required to use the training dataset to identify patterns that predict “potential” defaulters.

#### Expected Submission
Submissions should be made in the same format as the Sample Notebook provided. Train/Test split should be 80% for training & 20% for testing.

#### Evaluation
Submissions will be evaluated on the basis of roc_auc_score on 20% of train_dataset.

#### NOTE
The test_dataset was just a part of the Hackathon, the notebook you will be submitting should only train & test on train_data and predict a higher score.
#### About Data
- ID: Id of the user(All Unique)
- Income: Income of the user
- Age: Age of the user
- Experience: Professional experience of the user in years
- Profession: Profession of the user
- Married/Single: Whether married or not
- House_Ownership: Owned or rented or neither
- Car_Ownership: Does the person own a car
- STATE: State of residence
- CITY: City of residence
- CURRENT_JOB_YRS: Years of experience in the current job
- CURRENT_HOUSE_YRS: Number of years in the current residence\
- Risk_Flag: Defaulted on a loan(Target variable)

#### Importing necessary library and set options

In [ ]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from scipy.stats import chi2_contingency
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
%matplotlib inline
sns.set_theme(color_codes=True, style='darkgrid', 
              palette='deep', font='sans-serif')

## Importing Data and Data Cleansing

In [ ]:
df = pd.read_csv("../input/loan-prediction-based-on-customer-behavior/Training Data.csv")
df.head()

Exploring the data data type of each columns

In [ ]:
df.CITY.value_counts()

In [ ]:
df.info()

In [ ]:
df.describe()

Checking for null values

In [ ]:
df.isnull().sum()

There are no missing value in the datasets!

### Data Visualization 

### Distribution of Age

In [ ]:
sns.distplot(a=df["Age"]);

### Effect of House ownership on Risk Flag

In [ ]:
sns.countplot(x='House_Ownership', hue='Risk_Flag', data=df);

### Effect of Car owners on Risk Flag

In [ ]:
sns.countplot(x='Car_Ownership', hue='Risk_Flag', data=df);

### Effect of Marital Status on Risk Flag

In [ ]:
sns.countplot(x='Married/Single', hue='Risk_Flag', data=df);

### Distribution of Income

In [ ]:
sns.distplot(a=df["Income"]);

### Relationship of Numerical variable on target variable

In [ ]:
sns.heatmap(df.corr(), annot=True);

### Checking for Outliers

In [ ]:
sns.boxplot(y = 'Age', data = df);

In [ ]:
sns.boxplot(y = 'Income', data = df);

In [ ]:
r = df.groupby('Risk_Flag')['Risk_Flag'].count()
plt.pie(r, explode=[0.05, 0.1], labels=['Non-Defaulter', 'Defaulter'], radius=1.5, autopct='%1.1f%%',  shadow=True);

In [ ]:
print(len(df.Profession.unique()))
print(len(df.STATE.unique()))
print(len(df.CITY.unique()))

### Summary on Data Visualization
- Class 0 represents 88.00% of the dataset, while class 1 only 12.00%. The classes are heavily skewed we need to solve this issue
- There are no outliers in datasets. But we need to scale Age and Income
- Strong correlation between Experience and CURRENT_JOB_YRS May drop one column during feature selection process or use Principal Component Analysis (PCA)
- Married/Single House_Ownership Car_Ownership can be binarised or one-hot encoded
- We can find the relationship between target variable and categorical variable using Chi-square test

### Feature Engineering

Helping function for hypothesis testing

In [ ]:
def chi_square_test(data):
    stat, p, dof, expected = chi2_contingency(data)
    alpha = 0.05
    print("p value is " + str(p))
    if p <= alpha:
        print('Dependent (reject H0)')
    else:
        print('Independent (H0 holds true)')

### Chi Square Test

In [ ]:
car_ownership_risk_flag = pd.crosstab(df["Car_Ownership"], df["Risk_Flag"])
car_ownership_risk_flag

In [ ]:
chi_square_test(car_ownership_risk_flag)

In [ ]:
marital_status_risk_flag = pd.crosstab(df["Married/Single"], df["Risk_Flag"])
marital_status_risk_flag

In [ ]:
chi_square_test(marital_status_risk_flag)

In [ ]:
house_ownership_risk_flag = pd.crosstab(df["House_Ownership"], df["Risk_Flag"])
house_ownership_risk_flag

In [ ]:
chi_square_test(house_ownership_risk_flag)

### Performing Principal Component Analysis on CURRENT_JOB_YRS and Experience

In [ ]:
features = ["CURRENT_JOB_YRS","Experience"]

df_for_pca = df[features]
scaled_df_for_pca = (df_for_pca - df_for_pca.mean(axis=0))/df_for_pca.std()
scaled_df_for_pca

In [ ]:
pca = PCA()
df_pca = pca.fit_transform(scaled_df_for_pca)
component_names = [f"PC{i+1}" for i in range(df_pca.shape[1])]
df_pca = pd.DataFrame(df_pca, columns=component_names)

df_pca.head()

In [ ]:
df1 = pd.concat([df,df_pca],axis=1)
df1.head()

Label encoding for categorical variables

In [ ]:
features = ['Married/Single','Car_Ownership','Profession','CITY','STATE']
label_encoder = LabelEncoder()

for col in features:
    df1[col] = label_encoder.fit_transform(df1[col])

In [ ]:
df2 = pd.get_dummies(df1, columns = ["House_Ownership"])
df2.drop(["Id"],axis=1,inplace=True)

In [ ]:
X = df2.drop(['Risk_Flag'],axis=1)
y = df2.Risk_Flag
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 0)

In [ ]:
sm = SMOTE(random_state = 500)
X_res, y_res = sm.fit_resample(X_train, y_train)

Now the data is ready for implementation of Machine Learning model!!<br>
Since the target variable is either 0 or 1, So we will use ml models which is suitable for binary classification

### Machine learning model for Binary classification 

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter = 500000)
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

### KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5,metric='minkowski',p=2)
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

### Random Forest Classification

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(criterion='gini', bootstrap=True, random_state=420)
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(criterion="entropy",random_state=420)
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

### XGBoost

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(learning_rate=0.1,n_estimators=1000,use_label_encoder=False,random_state=420)
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

### AdaBoost Classifier

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier(random_state=15000)
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

### Passive Aggressive Classifier

In [ ]:
from sklearn.linear_model import PassiveAggressiveClassifier

model = PassiveAggressiveClassifier(random_state=14500)
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
from sklearn.ensemble import BaggingClassifier

model = BaggingClassifier(random_state=14500)
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

### Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier()
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

### Extra Tree

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

model = ExtraTreesClassifier(criterion="entropy",random_state=15000)
model.fit(X_res, y_res)
y_pred = model.predict(X_test)
accuracy = model.score(X_test, y_test)
accuracy

In [ ]:
print(classification_report(y_test,y_pred))

### Conclusion
XGBoost is the best suitable model for our dataset with accuracy 89%, Alternatively Random Forest and Extra Tree Classifier can also consider, since they gives accuracy of 88%.